# Fiche 4 — Valider sur des pays jamais vus (validation croisée groupée)

| | |
|---|---|
| 🎯 **Objectif** | Expliquer pourquoi un découpage aléatoire surestime un modèle, et utiliser `GroupKFold` pour mesurer l'erreur sur des groupes nouveaux |
| ⏱️ **Durée** | 45 min |
| 🧰 **Prérequis** | Fiche 1 ; train/test, validation croisée |
| 🔗 **Dans le projet** | Étape 2 (évaluation, éviter le surapprentissage) |

> **L'idée clé.** On évalue un modèle dans la situation où il servira. Ici : un **nouveau pays**. Si les années d'un même pays sont à la fois dans l'entraînement et dans le test, le modèle peut tricher en le reconnaissant.

In [1]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GroupKFold, KFold, cross_val_score

from agrireco import io, prepare

_, fao_dir = io.download()
# ⚠️ Cette cellule utilise la fusion du projet (agrireco.prepare) : construis d'abord la tienne (fiches 1 et 2)
fao, _ = prepare.build_fao_table(*prepare.load_fao_raw(fao_dir))
df = prepare.add_model_features(fao)
# get_dummies : une colonne 0/1 par culture (encodage « one-hot »)
X = pd.get_dummies(df[["crop", "avg_temp", "rainfall_mm", "log10_pesticides_t"]], dtype=float)
y = np.log(df["yield_t_ha"])  # on modélise le log du rendement (écarts de 1 à 13 entre cultures)

## 1. Pourquoi un pays se « reconnaît »

In [2]:
print("valeurs de pluie différentes par pays :", df.groupby("country")["rainfall_mm"].nunique().max())
df[df["country"] == "France"][["year", "crop", "rainfall_mm", "pesticides_t", "avg_temp"]].head(4)

valeurs de pluie différentes par pays : 1


,year,crop,rainfall_mm,pesticides_t,avg_temp
4822,1990,Maize,867.0,97701.0,11.96
4823,1991,Maize,867.0,103434.0,10.60
4824,1992,Maize,867.0,85249.0,11.15
4825,1993,Maize,867.0,91953.0,10.70


La pluie est **la même** pour toutes les années d'un pays : c'est une étiquette du pays. Avec la pluie et les
pesticides, un modèle flexible peut identifier le pays… et ressortir ses rendements passés.

## 2. Démonstration : le même modèle, deux façons de le valider

In [3]:
foret = RandomForestRegressor(n_estimators=100, random_state=0, n_jobs=-1)
aleatoire = -cross_val_score(foret, X, y, cv=KFold(5, shuffle=True, random_state=0),
                             scoring="neg_root_mean_squared_error")
par_pays = -cross_val_score(foret, X, y, cv=GroupKFold(5), groups=df["country"],
                            scoring="neg_root_mean_squared_error")
print(f"RMSE du log — validation aléatoire : {aleatoire.mean():.2f}")
print(f"RMSE du log — validation par pays  : {par_pays.mean():.2f}")

RMSE du log — validation aléatoire : 0.28
RMSE du log — validation par pays  : 0.72


## ⚠️ Piège : la validation aléatoire mesure la mémoire
Avec `KFold` (lignes mélangées), la forêt paraît excellente ; avec `GroupKFold` (un pays entier d'un seul côté),
l'erreur est **plus de deux fois plus grande**. Le premier chiffre est celui qu'on obtiendrait sur des pays déjà
connus. Le second est celui qui compte pour un nouveau client.

> **Lire une RMSE du log.** Une erreur de 0,1 ≈ 10 % d'erreur relative. 0,3 ≈ 35 %, 0,8 ≈ un facteur 2,2 entre
> prédiction et réalité.

`GroupKFold` découpe les **groupes** (ici les pays) en 5 paquets : chaque pays est en validation exactement une fois,
et jamais en même temps à l'entraînement.

In [4]:
cv = GroupKFold(5)
for i, (tr, va) in enumerate(cv.split(X, y, groups=df["country"])):
    commun = set(df["country"].iloc[tr]) & set(df["country"].iloc[va])
    print(f"pli {i} : {df['country'].iloc[va].nunique()} pays en validation, pays communs avec l'entraînement : {len(commun)}")

pli 0 : 24 pays en validation, pays communs avec l'entraînement : 0
pli 1 : 24 pays en validation, pays communs avec l'entraînement : 0
pli 2 : 23 pays en validation, pays communs avec l'entraînement : 0
pli 3 : 24 pays en validation, pays communs avec l'entraînement : 0
pli 4 : 24 pays en validation, pays communs avec l'entraînement : 0


### ✍️ À toi de jouer

Refais la comparaison avec une **régression linéaire** (`sklearn.linear_model.Ridge`). L'écart entre validation aléatoire et par pays est-il aussi grand ? Pourquoi ?

In [5]:
# Ta réponse ici

<details>
<summary><b>💡 Voir une solution</b> (après avoir cherché !)</summary>

```python
from sklearn.linear_model import Ridge

for nom, cv, g in [("aléatoire", KFold(5, shuffle=True, random_state=0), None), ("par pays", GroupKFold(5), df["country"])]:
    s = -cross_val_score(Ridge(), X, y, cv=cv, groups=g, scoring="neg_root_mean_squared_error")
    print(nom, round(s.mean(), 2))
```

**Pourquoi ?** l'écart est faible : une droite ne peut pas « apprendre par cœur » chaque pays, elle n'a que quelques coefficients. Plus un modèle est flexible, plus il peut mémoriser, et plus la validation aléatoire le flatte. C'est une raison de toujours comparer les modèles avec la validation qui correspond à l'usage.
</details>

## ✅ À retenir

1. On choisit la validation selon l'usage : ici, prédire pour un pays jamais vu.
2. `GroupKFold(groups=pays)` : un pays entier est d'un seul côté de chaque découpage.
3. Une validation aléatoire sur des données groupées mesure la mémoire du modèle, pas sa généralisation.
4. Plus un modèle est flexible, plus l'écart entre les deux validations est grand.
5. Le même découpage (les mêmes plis) sert à comparer tous les modèles.